In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


In [2]:
# Load CSV files for each class
ai_file = 'D:\RESEARCH 2024\40Hz ALZIEMER 2024\PROCESSED DATA\merged_AD.csv'
mci_file = 'D:\RESEARCH 2024\40Hz ALZIEMER 2024\PROCESSED DATA\merged_MCI.csv'
hc_file = 'D:\RESEARCH 2024\40Hz ALZIEMER 2024\PROCESSED DATA\merged_HC.csv'

# Load datasets
ai_data = pd.read_csv(ai_file)
mci_data = pd.read_csv(mci_file)
hc_data = pd.read_csv(hc_file)

# Drop 'time' column if it exists
for data in [ai_data, mci_data, hc_data]:l
    if 'time' in data.columns:
        data.drop(columns=['time'], inplace=True)

# Combine datasets
combined_data = pd.concat([ai_data, mci_data, hc_data], ignore_index=True)

# Separate features and labels
X = combined_data.drop(columns=['target'])
y = combined_data['target']

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Normalize features
X = (X - X.min()) / (X.max() - X.min())

# Convert data to PyTorch tensors
X_tensor = torch.tensor(X.values, dtype=torch.float32)
y_tensor = torch.tensor(y_encoded, dtype=torch.long)


In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch

# File paths
ai_file = r'D:\RESEARCH 2024\40Hz ALZIEMER 2024\PROCESSED DATA\merged_AD.csv'
mci_file = r'D:\RESEARCH 2024\40Hz ALZIEMER 2024\PROCESSED DATA\merged_MCI.csv'
hc_file = r'D:\RESEARCH 2024\40Hz ALZIEMER 2024\PROCESSED DATA\merged_HC.csv'

# Load datasets
ai_data = pd.read_csv(ai_file)
mci_data = pd.read_csv(mci_file)
hc_data = pd.read_csv(hc_file)

# Drop 'time' column if it exists
for data in [ai_data, mci_data, hc_data]:
    if 'time' in data.columns:
        data.drop(columns=['time'], inplace=True)

# Combine datasets
combined_data = pd.concat([ai_data, mci_data, hc_data], ignore_index=True)

# Separate features and target
X = combined_data.drop(columns=['target'])
y = combined_data['target']

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Min-Max normalization
X_normalized = (X - X.min()) / (X.max() - X.min())

# Merge normalized features with target for saving
normalized_data = X_normalized.copy()
normalized_data['target'] = y  # or use y_encoded if you prefer numeric labels

# Save normalized dataset
output_path = r'D:\RESEARCH 2024\40Hz ALZIEMER 2024\PROCESSED DATA\merged_normalized.csv'
normalized_data.to_csv(output_path, index=False)

print(f"Normalized dataset saved at:\n{output_path}")

# Optional: convert to PyTorch tensors
X_tensor = torch.tensor(X_normalized.values, dtype=torch.float32)
y_tensor = torch.tensor(y_encoded, dtype=torch.long)


Normalized dataset saved at:
D:\RESEARCH 2024\40Hz ALZIEMER 2024\PROCESSED DATA\merged_normalized.csv


In [3]:
# Define Generator and Discriminator for CGAN
class Generator(nn.Module):
    def __init__(self, latent_dim, label_dim, output_dim):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim + label_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim),
            nn.Tanh()
        )
    
    def forward(self, noise, labels):
        input = torch.cat((noise, labels), dim=1)
        return self.model(input)

class Discriminator(nn.Module):
    def __init__(self, input_dim, label_dim):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim + label_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    
    def forward(self, data, labels):
        input = torch.cat((data, labels), dim=1)
        return self.model(input)

In [4]:
# Hyperparameters
latent_dim = 100
n_features = X.shape[1]
n_classes = len(np.unique(y_encoded))
batch_size = 64
num_epochs = 500
learning_rate = 0.0002

# Create generator and discriminator
generator = Generator(latent_dim=latent_dim, label_dim=n_classes, output_dim=n_features)
discriminator = Discriminator(input_dim=n_features, label_dim=n_classes)

# Loss function and optimizers
loss_fn = nn.BCELoss()
optimizer_G = optim.Adam(generator.parameters(), lr=learning_rate)
optimizer_D = optim.Adam(discriminator.parameters(), lr=learning_rate)

# Helper function to create one-hot encoded labels
def one_hot_encode(labels, num_classes):
    return torch.eye(num_classes)[labels]

# Helper function to create random noise (latent vector)
def generate_latent_vector(batch_size, latent_dim):
    return torch.randn(batch_size, latent_dim)

# Training CGAN
for epoch in range(num_epochs):
    for i in range(0, len(X_tensor), batch_size):
        # Prepare real data and labels
        real_data = X_tensor[i:i+batch_size]
        real_labels = y_tensor[i:i+batch_size]
        real_one_hot_labels = one_hot_encode(real_labels, n_classes)
        
        # Real data labels for discriminator (1 for real)
        real_label_targets = torch.ones(real_data.size(0), 1)
        
        # Generate fake data
        z = generate_latent_vector(real_data.size(0), latent_dim)
        fake_one_hot_labels = one_hot_encode(real_labels, n_classes)  # Generate data with same labels
        fake_data = generator(z, fake_one_hot_labels)
        
        # Fake data labels for discriminator (0 for fake)
        fake_label_targets = torch.zeros(fake_data.size(0), 1)
        
        # Train Discriminator
        optimizer_D.zero_grad()
        real_loss = loss_fn(discriminator(real_data, real_one_hot_labels), real_label_targets)
        fake_loss = loss_fn(discriminator(fake_data.detach(), fake_one_hot_labels), fake_label_targets)
        d_loss = real_loss + fake_loss
        d_loss.backward()
        optimizer_D.step()
        
        # Train Generator
        optimizer_G.zero_grad()
        fake_data = generator(z, fake_one_hot_labels)
        g_loss = loss_fn(discriminator(fake_data, fake_one_hot_labels), real_label_targets)  # Want to fool discriminator
        g_loss.backward()
        optimizer_G.step()

    if epoch % 50 == 0:
        print(f"Epoch [{epoch}/{num_epochs}]  Loss D: {d_loss:.4f}, Loss G: {g_loss:.4f}")

# Generate synthetic samples for both MCI and HC classes
def generate_synthetic_samples(num_samples, latent_dim, generator, class_label, n_classes):
    z = generate_latent_vector(num_samples, latent_dim)
    class_one_hot_labels = one_hot_encode(torch.tensor([class_label] * num_samples), n_classes)
    synthetic_data = generator(z, class_one_hot_labels).detach().numpy()
    return synthetic_data

Epoch [0/500]  Loss D: 1.2655, Loss G: 0.7390
Epoch [50/500]  Loss D: 1.4279, Loss G: 0.6503
Epoch [100/500]  Loss D: 1.3231, Loss G: 0.7269
Epoch [150/500]  Loss D: 1.3895, Loss G: 0.7398
Epoch [200/500]  Loss D: 1.3658, Loss G: 0.7161
Epoch [250/500]  Loss D: 1.3327, Loss G: 0.7330
Epoch [300/500]  Loss D: 1.4592, Loss G: 0.6876
Epoch [350/500]  Loss D: 1.3443, Loss G: 0.7054
Epoch [400/500]  Loss D: 1.2371, Loss G: 0.7860
Epoch [450/500]  Loss D: 1.4572, Loss G: 0.6504


In [8]:
# Calculate the number of samples for each class
ai_count = len(ai_data)
mci_count = len(mci_data)
hc_count = len(hc_data)

# Calculate how many samples to generate for MCI and HC
num_synthetic_mci_samples = ai_count - mci_count  # To match AI class
num_synthetic_hc_samples = ai_count - hc_count  # To match AI class

# Generate synthetic data for MCI class
mci_class_label = label_encoder.transform(['MCI'])[0]
synthetic_mci_data = generate_synthetic_samples(num_synthetic_mci_samples, latent_dim, generator, mci_class_label, n_classes)

# Generate synthetic data for HC class
hc_class_label = label_encoder.transform(['HC'])[0]
synthetic_hc_data = generate_synthetic_samples(num_synthetic_hc_samples, latent_dim, generator, hc_class_label, n_classes)

# Combine original and synthetic data
X_balanced = np.vstack([X_normalized, synthetic_mci_data, synthetic_hc_data])
y_balanced = np.hstack([y_encoded, 
                         np.full(num_synthetic_mci_samples, mci_class_label),
                         np.full(num_synthetic_hc_samples, hc_class_label)])

print(f"Original AD size: {ai_count}")
print(f"Original MCI size: {mci_count}")
print(f"Synthetic MCI samples generated: {synthetic_mci_data.shape[0]}")
print(f"Original HC size: {hc_count}")
print(f"Synthetic HC samples generated: {synthetic_hc_data.shape[0]}")
print(f"Balanced dataset size: {X_balanced.shape[0]}")

Original AD size: 18000
Original MCI size: 6000
Synthetic MCI samples generated: 12000
Original HC size: 10000
Synthetic HC samples generated: 8000
Balanced dataset size: 54000


In [9]:
# Combine original and synthetic data
X_balanced = np.vstack([X_normalized, synthetic_mci_data, synthetic_hc_data])
y_balanced = np.hstack([y_encoded, 
                         np.full(num_synthetic_mci_samples, mci_class_label),
                         np.full(num_synthetic_hc_samples, hc_class_label)])

# Create a DataFrame for the balanced dataset
balanced_df = pd.DataFrame(X_balanced, columns=X_normalized.columns)
balanced_df['target'] = label_encoder.inverse_transform(y_balanced)

# Save the balanced dataset to a CSV file
balanced_file_path = 'D:\\RESEARCH 2024\\40Hz ALZIEMER 2024\\PROCESSED DATA\\balanced_dataset1.csv'
balanced_df.to_csv(balanced_file_path, index=False)

print(f"Balanced dataset saved to {balanced_file_path}")


Balanced dataset saved to D:\RESEARCH 2024\40Hz ALZIEMER 2024\PROCESSED DATA\balanced_dataset1.csv


In [ ]:
print(f"Original AD size: {ai_count}")
print(f"Original MCI size: {mci_count}")
print(f"Original MCI size: {hc_count}")